# SuccessApp — Phase 5: Get the on-device model (`.task`)

**What this notebook does:** downloads Google's **pre-quantized** Gemma 3n E2B LiteRT model — the ready-to-run `.task` file the Android app needs — and gets it onto your laptop reliably.

**Why we don't quantize ourselves:** MediaPipe's `convert_checkpoint` API fails on both Gemma 4 and Gemma 3n with the current `mediapipe` PyPI release (an internal `number_bits` assertion — the converter has no quantization recipe for these architectures yet). We use the `.task` file Google already built and validated. For a hackathon submission this is *more* credible, not less.

**Model split for the project (documented in `docs/submission_writeup.md`):**
- Colab notebooks 01 (triage + tools) and 02 (multimodal) run **Gemma 4 E2B** — that's the model showcase.
- The Android app runs **Gemma 3n E2B** (pre-quantized LiteRT) — because MediaPipe supports it on-device today.

**Runtime:** any Colab runtime works (CPU is fine — this notebook only downloads files). T4 not required.

## Step 0 — Install

In [ ]:
!pip install -q -U kagglehub kaggle

## Step 1 — Authenticate with Kaggle

Colab's built-in Kaggle integration uses an internal proxy that does not carry your license consent reliably. The robust method is to upload your `kaggle.json` directly.

**Get a fresh `kaggle.json`:**
1. Log into Kaggle in your browser with the account that has accepted the Gemma 3n license.
2. Kaggle -> Settings -> API -> **Expire API Token**, then **Create New Token** -> downloads `kaggle.json`.
3. Run the cell below and upload that file.

> The `kaggle.json` is a credentials file. It stays only in this Colab runtime — it is not written into the notebook.

In [ ]:
from google.colab import files
import os, json

print('Upload the kaggle.json you just downloaded:')
uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise SystemExit('You must upload a file named exactly kaggle.json')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

creds = json.load(open('/root/.kaggle/kaggle.json'))
assert 'username' in creds and 'key' in creds, 'kaggle.json missing username/key'
print(f"kaggle.json installed for user: {creds['username']}")

## Step 2 — (Optional) List available variations

Step 3 already tries the most likely handles automatically. Run this only if Step 3 fails and you need to find the exact slug. Otherwise skip straight to Step 3.

If the CLI listing is unhelpful, open **https://www.kaggle.com/models/google/gemma-3n** -> **Model Variations** tab -> look under the **tfLite** framework.

In [ ]:
!kaggle models instances list google/gemma-3n 2>/dev/null || echo '(CLI listing unavailable - use the website)'

## Step 3 — Download the pre-quantized model

The cell tries the most likely handle slugs automatically. If none work, open
kaggle.com/models/google/gemma-3n, find the exact tfLite variation, and put it as the
**first item** in `CANDIDATE_HANDLES`, then re-run.

In [ ]:
import kagglehub, os

# Most likely handles for the pre-quantized Gemma 3n E2B LiteRT model.
# If all fail, paste the exact handle from kaggle.com/models/google/gemma-3n as the
# FIRST item in this list and re-run.
CANDIDATE_HANDLES = [
    'google/gemma-3n/tfLite/gemma-3n-e2b-it-int4',
    'google/gemma-3n/tfLite/gemma-3n-E2B-it-int4',
    'google/gemma-3n/tfLite/gemma-3n-e2b-it',
    'google/gemma-3n/tflite/gemma-3n-e2b-it-int4',
]

MODEL_DIR = None
used_handle = None
for handle in CANDIDATE_HANDLES:
    try:
        print(f'Trying: {handle} ...')
        MODEL_DIR = kagglehub.model_download(handle)
        used_handle = handle
        print(f'  SUCCESS -> {MODEL_DIR}')
        break
    except Exception as e:
        print(f'  failed: {type(e).__name__}: {str(e)[:120]}')

if MODEL_DIR is None:
    raise SystemExit(
        'None of the candidate handles worked. Open kaggle.com/models/google/gemma-3n, '
        'find the exact tfLite variation slug, put it as the first item in CANDIDATE_HANDLES, '
        'and re-run this cell.'
    )

print()
print('Used handle:', used_handle)
print('Downloaded to:', MODEL_DIR)
print('Contents:')
for root, _, fnames in os.walk(MODEL_DIR):
    for fn in fnames:
        p = os.path.join(root, fn)
        print(f'  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')

## Step 4 — Locate and verify the model file

In [ ]:
import glob, shutil, os

# The download may contain a .task file directly, or a .tflite/.litertlm/.bin file.
candidates = []
for ext in ('*.task', '*.tflite', '*.litertlm', '*.bin'):
    candidates += glob.glob(os.path.join(MODEL_DIR, '**', ext), recursive=True)

if not candidates:
    raise SystemExit(
        f'No model file found under {MODEL_DIR}. '
        'Check the contents printed in Step 3 and pick the model file manually.'
    )

task_files = [c for c in candidates if c.endswith('.task')]
MODEL_FILE = task_files[0] if task_files else max(candidates, key=os.path.getsize)
size_mb = os.path.getsize(MODEL_FILE) / 1e6

FINAL_NAME = 'successapp-gemma3n-e2b-int4.task'
FINAL_PATH = f'/content/{FINAL_NAME}'
shutil.copy(MODEL_FILE, FINAL_PATH)

print(f'Model file:   {MODEL_FILE}')
print(f'Size:         {size_mb:.0f} MB')
print(f'Copied to:    {FINAL_PATH}')
print()
if size_mb < 500:
    print('WARNING: file is unusually small for a 2B-class model. Double-check you grabbed the model, not a config.')
elif size_mb > 8000:
    print('NOTE: file is large (>8 GB) - likely the int8 or unquantized variant. It still works; just bigger on the phone.')
else:
    print('Size looks right for an int4 2B-class model.')

## Step 5 — (Optional) Smoke test the model in Colab

Loads the `.task` with MediaPipe's LLM Inference API and runs one prompt. Optional — the real test is loading it in the Flutter app. If the API has drifted in your `mediapipe` version this cell may error; that does **not** mean the model file is bad. Skip to Step 6 if so.

In [ ]:
try:
    !pip install -q -U mediapipe
    from mediapipe.tasks.python.genai import llm_inference
    import inspect
    sig = inspect.signature(llm_inference.LlmInferenceOptions.__init__)
    print('LlmInferenceOptions accepts:', list(sig.parameters)[1:])
    opts = llm_inference.LlmInferenceOptions(model_path=FINAL_PATH, max_tokens=256)
    llm = llm_inference.LlmInference.create_from_options(opts)
    print()
    print('Model response:')
    print(llm.generate_response('Reply in one short sentence: what is your role in a wellbeing app?'))
except Exception as e:
    print(f'Smoke test skipped ({type(e).__name__}: {str(e)[:160]})')
    print('This is fine - the model file is still valid. The real test is in the Flutter app.')

## Step 6 — Get the file onto your laptop (split method)

The `.task` file is ~3 GB. Two transfer methods routinely fail in Colab:
- **Direct browser download** (`files.download` on the whole file) — stalls on multi-GB files.
- **Google Drive mount** — fails with `credential propagation was unsuccessful` when the browser blocks the OAuth handoff (privacy extensions, third-party cookie blocking).

So we **split the file into ~400 MB chunks** and download those — each chunk is small enough to download reliably, and no OAuth is involved. You recombine them on your laptop with one command.

Run the three cells below in order.

In [ ]:
# 6a — Split the .task file into 400 MB chunks
import os
os.chdir('/content')
!rm -f part_*
!split -b 400m {FINAL_NAME} part_
!ls -la /content/part_*

In [ ]:
# 6b — Generate the exact recombine command for your laptop. COPY the printed line.
import glob, os
parts = sorted(glob.glob('/content/part_*'))
names = [os.path.basename(p) for p in parts]
total_mb = sum(os.path.getsize(p) for p in parts) / 1e6
print(f'{len(names)} parts, {total_mb:.0f} MB total')
print()
print('=== WINDOWS: run this in your Downloads folder (cmd.exe) ===')
print('copy /b ' + ' + '.join(names) + ' successapp-gemma3n-e2b-int4.task')
print()
print('=== macOS / Linux: run this in your Downloads folder ===')
print('cat ' + ' '.join(names) + ' > successapp-gemma3n-e2b-int4.task')

In [ ]:
# 6c — Download every part. Your browser will save ~8 small files to Downloads.
# If a download is missed, just re-run this cell — already-downloaded parts re-download harmlessly.
from google.colab import files
import glob
for p in sorted(glob.glob('/content/part_*')):
    print('Downloading', os.path.basename(p))
    files.download(p)

## Step 7 — Recombine on your laptop, then wire into the Flutter app

1. Confirm all `part_*` files landed in your **Downloads** folder.
2. Open a terminal in Downloads and run the recombine command printed by cell 6b.
   - Windows (cmd.exe): `cd %USERPROFILE%\Downloads` then paste the `copy /b ...` line.
3. You now have `successapp-gemma3n-e2b-int4.task` (~3 GB). Verify the size matches Step 4.
4. Delete the `part_*` files.
5. Copy the `.task` into the app:
   ```
   mobile_app/android/app/src/main/assets/models/successapp-gemma3n-e2b-int4.task
   ```
6. `mobile_app/lib/services/gemma_service.dart` already references this exact filename — no code change needed.
7. Build: `cd mobile_app && flutter build apk --release`

That's the last Colab step. Everything else (Flutter build, demo video, submission) happens on your laptop.

## Appendix — Why not convert the model ourselves?

We tried `mediapipe.tasks.python.genai.converter.convert_checkpoint` on both Gemma 4 E2B and
Gemma 3n E2B. Both fail with:
```
AssertionError  (quantization_util.py: assert number_bits == 8 or number_bits == 4)
```
The converter in the current `mediapipe` PyPI release derives an invalid `number_bits` for a
layer in these architectures — it has no quantization recipe for them yet. This is a tooling
gap on Google's side, not a config error on ours. Using Google's officially pre-quantized
LiteRT release sidesteps it entirely and is the recommended path for on-device Gemma today.

**Alternative transfer methods (if the split method ever isn't needed):**
- Google Drive: `from google.colab import drive; drive.mount('/content/drive')` then
  `shutil.copy(FINAL_PATH, '/content/drive/MyDrive/')`. Only works if your browser allows the
  OAuth popup — it failed during development, which is why the split method is the default.